# Mini caso 4 — Asistente documental semántico

Este notebook plantea el mini caso de búsqueda semántica y respuesta asistida sobre el corpus documental RTVE 23-F.

El objetivo no es construir todavía un chatbot ni un sistema RAG completo, sino validar si el corpus permite desarrollar un asistente documental capaz de:

- recibir una consulta del usuario;
- localizar documentos o fragmentos relevantes;
- devolver evidencias trazables mediante `doc_id`, título y enlace al PDF;
- servir como base futura para una respuesta asistida con modelos de lenguaje.

La idea principal es transformar el corpus en un sistema de consulta semántica, evitando depender únicamente de búsquedas literales por palabras clave.

## 1. Carga de datos

Se utiliza como base el archivo limpio generado en la fase de limpieza general:

`data/processed/rtve_corpus_clean_base.csv`

Este archivo contiene el texto limpio (`text_clean_base`), metadatos básicos, información institucional y enlaces de trazabilidad.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = PROJECT_ROOT / "data" / "processed" / "rtve_corpus_clean_base.csv"

df = pd.read_csv(DATA_PATH)

print(f"Dataset cargado: {DATA_PATH.relative_to(PROJECT_ROOT)}")
print(f"Shape: {df.shape}")

df.head()

Dataset cargado: data/processed/rtve_corpus_clean_base.csv
Shape: (167, 25)


,doc_id,source_document_id,title,pages,detail_url,pdf_url,summary,keywords,text_full,text_clean_base,text_length_chars,text_length_words,text_clean_length_chars,text_clean_length_words,moncloa_id,moncloa_section,moncloa_subsection,final_match_status,coverage_moncloa,alpha_ratio,digit_ratio,uppercase_ratio,weird_char_ratio,n_title_years,title_main_year
0,rtve_1860,1860,Vista oral 2/81 del Consejo Supremo de Justicia Militar (20 de febrero de 1982).,3,https://23fbuscador.rtve.es/document/ocr/1860?page_size=200&page=1,https://www.rtve.es/contenidos/documentos/23f-desclasificado/99_1982_vista_oral_281_del_consejo_supremo_de_justicia_...,"El juicio oral 2/81 celebrado en febrero de 1982 se caracterizó por un intenso desarrollo en sus primeras sesiones, ...",C/SG/2820/20-02-82 DTOR. Vista oral 2/81,C/SG/2820/20-02-82\nDTOR.\n\nNOTA INFORMATIVA\n\nASUNTO: Vista oral 2/81\n\n1.- DESARROLLO DE LA SESIÓN CORRESPONDIE...,C/SG/2820/20-02-82\nDTOR.\n\nNOTA INFORMATIVA\n\nASUNTO: Vista oral 2/81\n\n1.- DESARROLLO DE LA SESIÓN CORRESPONDIE...,3934,640,3934,640,moncloa_0099,defensa,cni,high_confidence_match,True,0.777834,0.013726,0.147386,0.000000,1,1982.0
1,rtve_1859,1859,Vista oral 2/81 del Consejo Supremo de Justicia Militar (22 de febrero de 1982).,4,https://23fbuscador.rtve.es/document/ocr/1859?page_size=200&page=1,https://www.rtve.es/contenidos/documentos/23f-desclasificado/98_1982_vista_oral_281_del_consejo_supremo_de_justicia_...,Resumen global del documento:\n\nEl documento recoge el desarrollo de una serie de sesiones celebradas el 22 de febr...,C/SG/2896/22-02-82 Vista oral 2/81 Consejo Supremo de Justicia Militar,C/SG/2896/22-02-82\n\n# NOTA INFORMATIVA\n\nASUNTO: Vista oral 2/81. del Consejo Supremo de Justicia Militar.\n\n1.-...,C/SG/2896/22-02-82\n\n# NOTA INFORMATIVA\n\nASUNTO: Vista oral 2/81. del Consejo Supremo de Justicia Militar.\n\n1.-...,6417,1018,6417,1018,moncloa_0098,defensa,cni,high_confidence_match,True,0.781985,0.009506,0.195895,0.000156,1,1982.0
2,rtve_1858,1858,Vista oral 2/81 del Consejo Supremo de Justicia Militar (24 de febrero de 1982).,5,https://23fbuscador.rtve.es/document/ocr/1858?page_size=200&page=1,https://www.rtve.es/contenidos/documentos/23f-desclasificado/97_1982_vista_oral_281_del_consejo_supremo_de_justicia_...,Resumen global del documento:\n\nEl documento narra el desarrollo tenso y conflictivo de una serie de sesiones del C...,C/SG/2992/24-02-82 Vista Oral 2/81 Consejo Supremo de Justicia Militar,C/SG/2992/24-02-82\n\n# NOTA INFORMATIVA\n\nASUNTO: Vista Oral 2/81. del Consejo Supremo de Justicia Militar.\n\n## ...,C/SG/2992/24-02-82\n\n# NOTA INFORMATIVA\n\nASUNTO: Vista Oral 2/81. del Consejo Supremo de Justicia Militar.\n\n## ...,8183,1347,8183,1347,moncloa_0097,defensa,cni,high_confidence_match,True,0.784920,0.011487,0.124085,0.000611,1,1982.0
3,rtve_1857,1857,Vista oral 2/81 del Consejo Supremo de Justicia Militar (25 de febrero de 1982).,6,https://23fbuscador.rtve.es/document/ocr/1857?page_size=200&page=1,https://www.rtve.es/contenidos/documentos/23f-desclasificado/96_1982_vista_oral_281_del_consejo_supremo_de_justicia_...,El documento recoge el desarrollo de la sesión del Consejo Supremo de Justicia Militar en febrero de 1982 relativa a...,C/SG/3.081/25-02-82 Vista Oral 2/81 Consejo Supremo de Justicia Militar,C/SG/3.081/25-02-82\n\n# NOTA INFORMATIVA\n\nASUNTO: Vista Oral 2/81 del Consejo Supremo de Justicia Militar.\n\n## ...,C/SG/3.081/25-02-82\n\n# NOTA INFORMATIVA\n\nASUNTO: Vista Oral 2/81 del Consejo Supremo de Justicia Militar.\n\n## ...,11151,1826,11151,1826,moncloa_0096,defensa,cni,high_confidence_match,True,0.789257,0.008250,0.128167,0.000538,1,1982.0
4,rtve_1856,1856,Vista oral 2/81 del Consejo Supremo de Justicia Militar (26 de febrero de 1982).,6,https://23fbuscador.rtve.es/document/ocr/1856?page_size=200&page=1,https://www.rtve.es/contenidos/documentos/23f-desclasificado/95_1982_vista_oral_281_del_consejo_supremo_de_justicia_...,Resumen global del documento sobre la sesión d

## 2. Selección de campos útiles

Para un asistente documental no se necesita utilizar todas las columnas del dataset.

Los campos mínimos son:

- `doc_id`: identificador trazable del documento;
- `title`: título del documento;
- `text_clean_base`: texto limpio que servirá como base de búsqueda;
- `pdf_url`: enlace al documento original;
- `moncloa_section`: sección institucional, si está disponible;
- `text_clean_length_words`: longitud del texto limpio.

Estos campos permiten recuperar evidencia y mostrar al usuario de dónde procede cada resultado.

In [2]:
required_columns = [
    "doc_id",
    "title",
    "text_clean_base",
    "pdf_url",
    "moncloa_section",
    "text_clean_length_words",
]

missing_columns = [col for col in required_columns if col not in df.columns]

print("Columnas requeridas ausentes:", missing_columns)

df_assistant = df[required_columns].copy()

df_assistant.head()

Columnas requeridas ausentes: []


,doc_id,title,text_clean_base,pdf_url,moncloa_section,text_clean_length_words
0,rtve_1860,Vista oral 2/81 del Consejo Supremo de Justicia Militar (20 de febrero de 1982).,C/SG/2820/20-02-82\nDTOR.\n\nNOTA INFORMATIVA\n\nASUNTO: Vista oral 2/81\n\n1.- DESARROLLO DE LA SESIÓN CORRESPONDIE...,https://www.rtve.es/contenidos/documentos/23f-desclasificado/99_1982_vista_oral_281_del_consejo_supremo_de_justicia_...,defensa,640
1,rtve_1859,Vista oral 2/81 del Consejo Supremo de Justicia Militar (22 de febrero de 1982).,C/SG/2896/22-02-82\n\n# NOTA INFORMATIVA\n\nASUNTO: Vista oral 2/81. del Consejo Supremo de Justicia Militar.\n\n1.-...,https://www.rtve.es/contenidos/documentos/23f-desclasificado/98_1982_vista_oral_281_del_consejo_supremo_de_justicia_...,defensa,1018
2,rtve_1858,Vista oral 2/81 del Consejo Supremo de Justicia Militar (24 de febrero de 1982).,C/SG/2992/24-02-82\n\n# NOTA INFORMATIVA\n\nASUNTO: Vista Oral 2/81. del Consejo Supremo de Justicia Militar.\n\n## ...,https://www.rtve.es/contenidos/documentos/23f-desclasificado/97_1982_vista_oral_281_del_consejo_supremo_de_justicia_...,defensa,1347
3,rtve_1857,Vista oral 2/81 del Consejo Supremo de Justicia Militar (25 de febrero de 1982).,C/SG/3.081/25-02-82\n\n# NOTA INFORMATIVA\n\nASUNTO: Vista Oral 2/81 del Consejo Supremo de Justicia Militar.\n\n## ...,https://www.rtve.es/contenidos/documentos/23f-desclasificado/96_1982_vista_oral_281_del_consejo_supremo_de_justicia_...,defensa,1826
4,rtve_1856,Vista oral 2/81 del Consejo Supremo de Justicia Militar (26 de febrero de 1982).,"C/SG/3.249/26-02-82\nSG\n\n# NOTA INFORMATIVA\n\nASUNTO: Vista oral de la causa 2/81, del Consejo Supremo de Justici...",https://www.rtve.es/contenidos/documentos/23f-desclasificado/95_1982_vista_oral_281_del_consejo_supremo_de_justicia_...,defensa,1740


## 3. Validación de textos disponibles

Antes de plantear búsqueda semántica, se comprueba si todos los documentos tienen texto limpio disponible.

El sistema solo será viable si existe suficiente contenido textual para recuperar documentos o fragmentos relevantes.

In [3]:
text_availability = {
    "n_documents": len(df_assistant),
    "n_text_available": df_assistant["text_clean_base"].notna().sum(),
    "n_empty_text": (df_assistant["text_clean_base"].fillna("").str.strip() == "").sum(),
    "min_text_length_words": int(df_assistant["text_clean_length_words"].min()),
    "median_text_length_words": float(df_assistant["text_clean_length_words"].median()),
    "mean_text_length_words": float(df_assistant["text_clean_length_words"].mean()),
    "max_text_length_words": int(df_assistant["text_clean_length_words"].max()),
}

pd.DataFrame(text_availability.items(), columns=["check", "value"])

,check,value
0,n_documents,167.000000
1,n_text_available,167.000000
2,n_empty_text,0.000000
3,min_text_length_words,72.000000
4,median_text_length_words,579.000000
5,mean_text_length_words,2075.443114
6,max_text_length_words,95293.000000


## 4. Revisión de longitudes y necesidad de fragmentación

Un asistente documental puede trabajar a nivel documento completo o a nivel fragmento.

El EDA general ya mostró que el corpus es muy heterogéneo: hay documentos breves y documentos muy largos. Para preguntas concretas, trabajar con documentos completos puede ser poco preciso, porque un texto largo puede contener muchos temas distintos.

Por tanto, se revisa cuántos documentos superarían umbrales razonables de longitud y podrían necesitar fragmentación.

In [4]:
length_thresholds = [1000, 3000, 5000, 10000]

length_review = []

for threshold in length_thresholds:
    n_docs = (df_assistant["text_clean_length_words"] > threshold).sum()
    length_review.append({
        "threshold_words": threshold,
        "n_documents_above_threshold": int(n_docs),
        "percentage": round(n_docs / len(df_assistant) * 100, 2),
    })

df_length_review = pd.DataFrame(length_review)

df_length_review

,threshold_words,n_documents_above_threshold,percentage
0,1000,63,37.72
1,3000,18,10.78
2,5000,9,5.39
3,10000,6,3.59


In [5]:
df_long_documents = (
    df_assistant
    .sort_values("text_clean_length_words", ascending=False)
    .head(10)
    [["doc_id", "title", "moncloa_section", "text_clean_length_words", "pdf_url"]]
)

df_long_documents

,doc_id,title,moncloa_section,text_clean_length_words,pdf_url
161,rtve_1699,Transcripción de cintas grabadas con conversaciones telefónicas con varias personas intervenidas a la esposa de Tejero.,interior,95293,https://www.rtve.es/contenidos/documentos/23f-desclasificado/07_2026_transcripcion_de_cintas_grabadas_con_conversaci...
159,rtve_1701,Télex interiores y de agencias recibidos en 2ª sección EM el día 23-F informando del estado de situación (23 de febr...,interior,19857,https://www.rtve.es/contenidos/documentos/23f-desclasificado/09_1981_telex_interiores_y_de_agencias_recibidos_en_2%C...
63,rtve_1797,Investigación y declaraciones personal AOME por JDDI (9 de abril de 1981).,defensa,14485,https://www.rtve.es/contenidos/documentos/23f-desclasificado/36_1981_investigacion_y_declaraciones_personal_aome_por...
76,rtve_1784,Policía Nacional. Informe de situación. Marca: reservado-confidencial (12 de noviembre de 1981).,interior,13639,https://www.rtve.es/contenidos/documentos/23f-desclasificado/23_1981_policia_nacional_informe_de_situacion_marca_res...
74,rtve_1786,"""Juicio del 23-F: acotaciones al desarrollo del juicio",interior,11080,https://www.rtve.es/contenidos/documentos/23f-desclasificado/25_2026_juicio_del_23_f_acotaciones_al_desarrollo_del_j...
126,rtve_1734,"""Nota Informativa sobre la repercusión en prensa del arresto de Tejero en 1978 cuando era Jefe de la Comandancia de ...",interior,10253,https://www.rtve.es/contenidos/documentos/23f-desclasificado/12_1978_nota_informativa_sobre_la_repercusion_en_prensa...
50,rtve_1810,Relato de los sucesos de los días 23 y 24 de febrero.,defensa,8965,https://www.rtve.es/contenidos/documentos/23f-desclasificado/49_2026_relato_de_los_sucesos_de_los_dias_23_y_24_de_fe...
157,rtve_1703,Semestral de la amenaza interior (10 de febrero de 1981; 9 de marzo de 1981).,defensa,8714,https://www.rtve.es/contenidos/documentos/23f-desclasificado/101_1981_semestral_de_la_amenaza_interior_10_de_febrero...
93,rtve_1767,"""Informe de las distintas Jefaturas Superiores: Comisaría General de Información. """"Situación actual en las distinta...",interior,7189,https://www.rtve.es/contenidos/documentos/23f-desclasificado/15_1981_informe_de_las_distintas_jefaturas_superiores_c...
163,rtve_1697,"""Documentación con una presunta planificación del golpe",interior,4882,https://www.rtve.es/contenidos/documentos/23f-desclasificado/05_1980_documentacion_con_una_presunta_planificacion_de...


**Lectura del bloque**

La revisión de longitudes confirma que la fragmentación será necesaria para una parte del corpus.

Hay 63 documentos con más de 1.000 palabras, 18 con más de 3.000 palabras, 9 con más de 5.000 palabras y 6 con más de 10.000 palabras. Además, el documento más largo alcanza más de 95.000 palabras.

Esto significa que una búsqueda a nivel de documento completo podría ser demasiado imprecisa en textos largos: el sistema recuperaría el documento correcto, pero no necesariamente la parte concreta que responde a la pregunta.

Por ello, la metodología futura deberá trabajar preferentemente con fragmentos o `chunks`, manteniendo siempre la trazabilidad al documento original mediante `doc_id`, título y enlace al PDF.

## 5. Planteamiento del sistema

El mini caso se plantea como un asistente documental semántico, no como un chatbot generalista.

El sistema tendría dos funcionalidades principales:

1. **Búsqueda por pregunta**  
   El usuario formula una pregunta y el sistema devuelve los documentos o fragmentos más relevantes.

2. **Recomendación de documentos relacionados**  
   Dado un documento inicial, el sistema recomienda otros documentos similares o complementarios.

Ambas funcionalidades comparten una lógica común: representar documentos o fragmentos en un espacio vectorial y recuperar los elementos más cercanos.

In [6]:
example_queries = [
    "¿Qué documentos mencionan al CESID?",
    "¿Qué aparece sobre Tejero en las vistas orales?",
    "¿Qué documentos hablan de la Guardia Civil?",
    "¿Qué documentos tratan sobre la sentencia?",
    "¿Qué actores institucionales aparecen en documentos de Defensa?",
]

pd.DataFrame({"example_query": example_queries})

,example_query
0,¿Qué documentos mencionan al CESID?
1,¿Qué aparece sobre Tejero en las vistas orales?
2,¿Qué documentos hablan de la Guardia Civil?
3,¿Qué documentos tratan sobre la sentencia?
4,¿Qué actores institucionales aparecen en documentos de Defensa?


## 6. Metodología futura

Para la entrega final se plantea el siguiente flujo:

1. Seleccionar el texto limpio (`text_clean_base`) como base documental.
2. Dividir documentos largos en fragmentos si superan un umbral de longitud.
3. Representar documentos o fragmentos mediante:
   - TF-IDF como baseline interpretable;
   - embeddings semánticos como mejora posterior.
4. Para una consulta del usuario, calcular su representación vectorial.
5. Recuperar los `top-k` documentos o fragmentos más similares.
6. Mostrar resultados con evidencia:
   - fragmento recuperado;
   - `doc_id`;
   - título;
   - enlace al PDF.
7. En una versión ampliada, usar un LLM para redactar una respuesta basada únicamente en los fragmentos recuperados.

La versión inicial puede funcionar como buscador semántico sin generación automática de texto. La generación con LLM se considera una extensión posterior.

In [7]:
planned_system = {
    "input": "consulta del usuario o documento inicial",
    "base_text": "text_clean_base",
    "retrieval_unit": "documento completo o fragmento",
    "baseline_representation": "TF-IDF",
    "advanced_representation": "embeddings semánticos",
    "retrieval_method": "similitud coseno top-k",
    "outputs": [
        "fragmentos/documentos relevantes",
        "doc_id",
        "title",
        "pdf_url",
        "score de similitud",
    ],
    "possible_extension": "respuesta asistida con LLM usando solo evidencias recuperadas",
}

planned_system

{'input': 'consulta del usuario o documento inicial',
 'base_text': 'text_clean_base',
 'retrieval_unit': 'documento completo o fragmento',
 'baseline_representation': 'TF-IDF',
 'advanced_representation': 'embeddings semánticos',
 'retrieval_method': 'similitud coseno top-k',
 'outputs': ['fragmentos/documentos relevantes',
  'doc_id',
  'title',
  'pdf_url',
  'score de similitud'],
 'possible_extension': 'respuesta asistida con LLM usando solo evidencias recuperadas'}

## 7. Riesgos y limitaciones

Los principales riesgos del mini caso son:

- documentos muy largos que requieran fragmentación;
- ruido OCR que afecte a la recuperación semántica;
- evaluación parcialmente cualitativa, al no existir un conjunto de preguntas con respuestas esperadas;
- posible solape entre documentos muy parecidos, especialmente en la serie de vistas orales;
- riesgo de respuestas no respaldadas si se usa un LLM generativo sin controlar las evidencias.

Para evitar estos problemas, el sistema deberá priorizar la trazabilidad. Toda respuesta o recomendación deberá mostrar los documentos o fragmentos usados como evidencia.

## 8. Conclusión de viabilidad

El mini caso es viable porque el corpus contiene 167 documentos con texto limpio disponible en `text_clean_base`, sin textos vacíos y con enlaces trazables a los PDFs originales. Esto permite construir un sistema de recuperación documental en el que el usuario formule una consulta y el sistema devuelva evidencias relevantes junto con `doc_id`, título, enlace al PDF y score de similitud.

La principal cuestión metodológica es la heterogeneidad en la longitud de los textos. Aunque la mediana es de 579 palabras y la longitud mínima es de 72, la media asciende a 2.075 palabras porque algunos documentos son muy extensos, llegando el mayor a superar las 95.000 palabras. Por tanto, el sistema puede partir del campo `text_clean_base`, pero la fase final deberá distinguir entre documentos cortos, que podrían recuperarse completos, y documentos largos, que deberían dividirse en fragmentos para mejorar la precisión de las búsquedas.

La versión inicial se plantea como un sistema de búsqueda semántica con recuperación de evidencias. La recomendación de documentos relacionados puede integrarse como funcionalidad complementaria usando la misma lógica de similitud vectorial.

La respuesta asistida mediante LLM debe considerarse una ampliación futura, no el punto de partida. Si se implementa, tendrá que estar limitada a respuestas basadas exclusivamente en fragmentos recuperados del corpus, para evitar conclusiones no justificadas por los documentos.